# Sanidad de las métricas: runs del piloto

Objetivo: comprobar, antes de ningún contraste de hipótesis, si las métricas de gradiente **tienen sentido** sobre las 24 runs del piloto de calibración. Todo se apoya en `src/analysis.py` (cálculo) y `src/plots.py` (figuras); aquí no se calcula nada, solo se llama.

1. **Validez y rangos**: NaN/Inf, columnas ausentes y valores fuera del rango teórico, más las identidades duras entre columnas.
2. **Movimiento**: si cada métrica se mueve de forma sistemática dentro de un run o solo tiembla de época a época.
3. **Dirección frente a la teoría**: si la trayectoria se mueve en el sentido que predice cada artículo.
4. **Redundancia**: qué métricas se mueven juntas, y el caso extremo de dos que son la misma cantidad.

**Alcance.** El piloto tiene **un run por celda**, así que no hay dispersión intra-celda que correlacionar contra la eficiencia: nada de este notebook toca el plan confirmatorio congelado (`docs/research/4 - Análisis.md`), que corre sobre la matriz en `reports/`. Las mismas funciones servirán luego para la matriz.

**Cautelas del piloto.**

- Las runs corren a **2x presupuesto**, hasta dentro del sobreajuste. `progress_frac` es relativo a ese 2x, así que el presupuesto congelado 1x equivale a `progress_frac` 0,5. Las predicciones de los artículos son sobre la fase de entrenamiento, no sobre la cola posterior a la meseta: por eso la dirección se mira también en ventana temprana.
- En `tiny_imagenet` los campos de test y de gap del `summary.json` están corruptos (bug anterior al arreglo); las métricas de la trayectoria y los campos de validación y de tiempo son válidos.
- Hay **una** run por celda, así que ninguna figura puede separar el efecto de la arquitectura del de la semilla. Cuando una figura agrupa por dataset, dentro de cada grupo hay seis configuraciones distintas, no seis repeticiones.

## Preparación

In [ ]:
import sys, pathlib
import pandas as pd

# Localiza la raíz del repo (carpeta con pyproject.toml) y añade src/ al path.
_p = pathlib.Path.cwd()
ROOT = next((q for q in [_p, *_p.parents] if (q / "pyproject.toml").exists()), _p)
sys.path.insert(0, str(ROOT / "src"))

import analysis as A      # backend sin ploteo: carga y diagnósticos
import plots as P         # capa de figuras: estilo único del TFG

P.use_thesis_style()

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)

traj = A.load_trajectories()        # por defecto reports_pilot/
summ = A.load_summaries()
print(f"{traj['run_name'].nunique()} runs, {len(traj)} filas época-run")
print(f"métricas conocidas: {len(A.metric_columns())} | titulares: {len(A.headline_columns())}")

## 1. Validez y rangos

`validity_report` da, por columna conocida: rango observado, conteos de NaN/Inf, valores fuera de cota y estado. `identity_report` comprueba invariantes deterministas que **deben** cumplirse fila a fila, de modo que cualquier violación es un fallo de implementación y no un hallazgo:

- `eta = -min_cos` (confusión de gradientes).
- `min_cos <= p05_cos <= median_cos` (ordenación de cuantiles del mismo conjunto de cosenos).
- `gsnr median <= p95`.
- `tse/cumulative` no decreciente (suma corrida de pérdidas no negativas).

In [ ]:
val = A.validity_report(traj)
print("Columnas fuera de 'ok':", (val["status"] != "ok").sum(), "de", len(val))
val

In [ ]:
A.identity_report(traj)

## 2. Movimiento: ¿señal o temblor?

Una métrica solo puede predecir algo si se mueve, y hay dos formas muy distintas de moverse: derivar de forma sistemática a lo largo del entrenamiento, o temblar de época a época alrededor de un valor fijo. Solo la primera es señal.

El estadístico es `signal_to_jitter = std(valores) / std(primeras diferencias)` dentro de cada run. Numerador y denominador escalan igual con la métrica, así que **el cociente no depende ni de las unidades ni de la escala**: multiplicar una métrica por un millón no lo cambia. Esa es exactamente la propiedad que hace falta aquí, porque la misma cantidad vive en escalas muy distintas según el dataset (la val loss va de 0,02 en MNIST a 71 en Tiny-ImageNet).

La referencia tampoco es un umbral elegido a mano. Una trayectoria que sea ruido blanco alrededor de una constante cumple `std(diff) = sqrt(2) * std(valores)`, así que su cociente vale exactamente 1/sqrt(2) ≈ 0,71. Es la línea de la figura. Muy por encima, la métrica deriva; pegada a ella, lo que parece movimiento es temblor de medición.

La columna `below_noise` es la comparación contra esa línea y nada más. No se llama *degenerada* a propósito: la referencia es el valor asintótico, así que una métrica que fuera puro ruido caería a un lado o a otro de la línea aproximadamente la mitad de las veces, y un veredicto binario sobre un solo run afirmaría más de lo que el estadístico aguanta. Lo que se lee es la distribución completa frente a la línea.

In [ ]:
deg = A.degeneracy_report(traj)
A.degeneracy_summary(deg).round(2)

In [ ]:
resumen = A.degeneracy_summary(deg)
fig = P.strip(deg, "signal_to_jitter", "key", order=list(resumen.index), log=True,
              xlabel="señal / temblor (escala logarítmica)",
              reference=A.NOISE_RATIO, reference_label="ruido blanco (1/√2)")
P.save(fig, "pilot-movimiento");

## 3. Dirección frente a la teoría

Para cada run se mide el Spearman entre el valor de la métrica y la época, y se compara con el signo que predice su artículo. El detalle está en la tabla `SPECS` de `analysis.py`; las métricas sin predicción direccional robusta (GSNR, disparidad de gradientes) se registran pero no se puntúan.

La figura usa `rho_signed = rho * signo esperado` y no el rho crudo. El motivo es que **siete de las métricas puntuadas esperan bajar y cuatro esperan subir**, así que un rho de −0,8 significa "concuerda" en una columna y "contradice" en la siguiente; sin el giro de signo las columnas no son comparables entre sí. Tras el giro, positivo significa siempre lo mismo: la métrica se comporta como predice su artículo.

Dos avisos de lectura. `tse/cumulative` queda fuera de la figura porque su tendencia positiva es una identidad matemática, no una predicción: es una suma corrida de pérdidas no negativas, así que su rho vale +1 en las 24 runs por construcción, y de hecho ya se verifica como identidad dura en la sección 1. Y `train_loss` y `val_acc` no son métricas bajo estudio sino **anclas de salud del run**: si no concuerdan, lo que falla es el entrenamiento, no la teoría.

In [ ]:
import pandas as pd

# ~ mitad del presupuesto 1x, que es donde viven las ventanas del plan.
td = pd.concat([
    A.trend_report(traj).assign(ventana="trayectoria completa"),
    A.trend_report(traj, progress_max=0.25).assign(ventana="ventana temprana"),
])
cmp = (A.trend_summary(A.trend_report(traj))[["family", "expected", "frac_agree", "median_rho_signed"]]
       .join(A.trend_summary(A.trend_report(traj, progress_max=0.25))
             [["frac_agree", "median_rho_signed"]], rsuffix="_early"))
cmp.round(3)

In [ ]:
# Ordenado por el propio eje de la figura: mediana de rho_signed en la
# trayectoria completa, de mayor a menor concordancia.
orden = [k for k in cmp.sort_values("median_rho_signed", ascending=False).index
         if k != "tse/cumulative"]
fig = P.strip(td[td["key"] != "tse/cumulative"], "rho_signed", "key",
              order=orden, panel_by="ventana", reference=0.0,
              reference_label="sin tendencia",
              xlabel="Spearman(valor, época) × signo esperado")
P.save(fig, "pilot-tendencia");

## 4. Redundancia entre métricas

`redundancy_matrix` promedia las matrices de Spearman intra-run sobre las métricas titulares (8 de gradiente más val_acc, val_loss y TSE-EMA). Promediar dentro de cada run evita que las diferencias de escala entre datasets fabriquen la correlación, que es la trampa de Simpson habitual en este tipo de mapa.

**No es un contraste confirmatorio** ni alimenta ninguna hipótesis: es la foto previa para decidir la poda de métricas redundantes.

Se dibuja solo el triángulo inferior. La matriz es simétrica, así que la mitad superior es la misma información repetida, y sobre todo la diagonal de unos es el valor máximo posible: al dejarla, ancla la escala de color y aplana todo lo demás.

In [ ]:
corr = A.redundancy_matrix(traj)
fig = P.heatmap(corr, cbar_label="Spearman intra-run promediado",
                vmin=-1, vmax=1, annot=True, mask_upper=True)
P.save(fig, "pilot-redundancia");

In [ ]:
A.top_redundant_pairs(corr, n=12).round(3)

## 5. El caso extremo: GNS = M/m-coherencia − 1

La tabla anterior da −1,00 exacto entre `noise_scale/simple` y `mcoh/global`. Un coeficiente de −1,00 sería compatible con cualquier relación monótona decreciente, así que por sí solo no demuestra que sean la misma cantidad. La comprobación honesta es dibujar una contra la otra: la diagonal solo la satisface la igualdad.

Y lo son: dos de las ocho métricas **no son dos señales**, sino la misma reparametrizada, con M el tamaño del conjunto de medición.

In [ ]:
M = 256   # tamaño del conjunto de medición (FIXED_KNOBS)
pred = M / traj["mcoh/global"] - 1
fig = P.identity_scatter(pred, traj["noise_scale/simple"],
                         xlabel="M / m-coherencia − 1", ylabel="escala de ruido (simple)",
                         log=True)
P.save(fig, "pilot-identidad-gns-mcoh")
err = ((pred - traj["noise_scale/simple"]).abs() / traj["noise_scale/simple"].abs()).max()
print(f"error relativo máximo: {err:.2e}")

## 6. Galería de trayectorias

Inspección visual final: cada métrica titular frente a `progress_frac`, una línea por run, coloreada por dataset. Es el "se mueven como deberían" a ojo, que complementa los números de arriba.

Los paneles de magnitudes no negativas y sin cota superior van en escala logarítmica. No es una preferencia estética: en escala lineal la val loss de MNIST (0,02 a 0,18) queda pegada al cero frente a la de Tiny-ImageNet (2,3 a 71), y el panel deja de decir nada sobre tres de los cuatro datasets. Qué panel entra lo decide el rango teórico declarado en `SPECS`, no la inspección de los datos.

No se agrega. Con seis runs por dataset, y siendo esas seis configuraciones distintas (tres arquitecturas por dos optimizadores) en vez de seis repeticiones, una banda intercuartílica mediría la diferencia entre arquitecturas y se leería como dispersión entre repeticiones. Sobre la matriz, con 40 runs por celda, sí tendrá sentido agregar.

In [ ]:
titulares = A.headline_columns()
# Magnitudes no negativas y sin cota superior: pueden abarcar órdenes de magnitud.
log_keys = {k for k in titulares
            if A.SPEC_BY_KEY[k].lo == 0.0 and A.SPEC_BY_KEY[k].hi is None}
fig = P.trajectory_grid(traj, titulares, color_by="dataset", ncols=3, log_keys=log_keys)
P.save(fig, "pilot-trayectorias")
sorted(log_keys)

## 7. Coste de la instrumentación

Cuánto cuesta medir por cada segundo de entrenamiento. La cifra absoluta no dice nada útil porque las celdas difieren en dos órdenes de magnitud de coste; lo que decide si la instrumentación completa es asumible es el **cociente**, y lo que lo gobierna es la arquitectura, porque el barrido por muestra es caro donde la última capa es grande. La línea marca la paridad, y a su derecha medir cuesta más que entrenar.

In [ ]:
coste = summ.assign(ratio=summ["metric_seconds"] / summ["train_seconds"])
fig = P.strip(coste, "ratio", "model", order=["cnn", "resnet18", "fc"],
              xlabel="segundos de medición por segundo de entrenamiento",
              reference=1.0, reference_label="paridad")
P.save(fig, "pilot-coste")
peor = coste.loc[coste["ratio"].idxmax()]
print(f"peor caso: {peor['run_name']} -> {peor['ratio']:.2f}x")
coste.groupby("model")["ratio"].describe()[["min", "50%", "max"]].round(2)

## 8. Síntesis

Apuntar aquí las conclusiones de "tienen sentido / hay que vigilar" tras correr las celdas: validez estructural, identidades, qué métricas concuerdan con su teoría, cuáles no se distinguen del temblor y los bloques de redundancia.

Estas observaciones son **descriptivas** del piloto; ninguna decisión confirmatoria se toma con estos datos.